# Module 11 — Dynamic Programming 2D Knapsack and Grids

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

import pytest   # some assertions check that an invalid input RAISES
sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_unique_paths import unique_paths
from p02_min_path_sum import min_path_sum
from p03_edit_distance import edit_distance

print("module 11: Dynamic Programming 2D Knapsack and Grids")
print("problems available:", 8)
for name in ['p01_unique_paths', 'p02_min_path_sum', 'p03_edit_distance', 'p04_knapsack_01', 'p05_can_partition', 'p06_lcs', 'p07_unique_paths_obstacles', 'p08_longest_palindromic_subseq']:
    print(f"  {name}")

## 1. Baseline — `p01_unique_paths`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
assert unique_paths(3, 7) == 28
assert unique_paths(3, 2) == 3
assert unique_paths(1, 1) == 1
# A single row or column has exactly one path.
assert unique_paths(1, 10) == 1
assert unique_paths(10, 1) == 1
assert unique_paths(2, 2) == 2
# Symmetric in its arguments.
assert unique_paths(3, 7) == unique_paths(7, 3)
with pytest.raises(ValueError):
    unique_paths(0, 5)
# It is a binomial coefficient: C(m+n-2, m-1).
import math
for m in range(1, 9):
    for n in range(1, 9):
        assert unique_paths(m, n) == math.comb(m + n - 2, m - 1), (m, n)

print("all assertions held")

## 2. Predict before you run

0/1 knapsack with one item of weight 1 and value 10, and a capacity of 3. Predict the answer. Then predict what a single-row implementation that iterates capacity upward returns instead.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
assert min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
assert min_path_sum([[1, 2, 3], [4, 5, 6]]) == 12
assert min_path_sum([[5]]) == 5
# A single row or column is just the total.
assert min_path_sum([[1, 2, 3]]) == 6
assert min_path_sum([[1], [2], [3]]) == 6
# Negative values are handled by the same recurrence.
assert min_path_sum([[1, -1], [-1, 1]]) == 1
with pytest.raises(ValueError):
    min_path_sum([])
# Cross-check against exhaustive path enumeration.
import itertools
def brute(g):
    rows, cols = len(g), len(g[0])
    best = None
    for moves in itertools.permutations('D' * (rows - 1) + 'R' * (cols - 1)):
        r = c = 0
        total = g[0][0]
        for mv in moves:
            if mv == 'D':
                r += 1
            else:
                c += 1
            total += g[r][c]
        best = total if best is None else min(best, total)
    return best if best is not None else g[0][0]
for g in ([[1, 3, 1], [1, 5, 1], [4, 2, 1]], [[2, 1], [1, 9]], [[1, 2, 3], [4, 5, 6]]):
    assert min_path_sum(g) == brute(g), g

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert edit_distance("horse", "ros") == 3
assert edit_distance("intention", "execution") == 5
# Empty strings.
assert edit_distance("", "") == 0
assert edit_distance("abc", "") == 3
assert edit_distance("", "abc") == 3
# Identical strings cost nothing.
assert edit_distance("same", "same") == 0
# A single operation of each kind.
assert edit_distance("cat", "cats") == 1
assert edit_distance("cats", "cat") == 1
assert edit_distance("cat", "bat") == 1
# Symmetric: insertions and deletions mirror each other.
for x, y in (("horse", "ros"), ("abc", "yabd"), ("kitten", "sitting")):
    assert edit_distance(x, y) == edit_distance(y, x), (x, y)
assert edit_distance("kitten", "sitting") == 3
# The distance never exceeds the longer length.
for x, y in (("abcdef", "uvwxyz"), ("a", "bcdefg")):
    assert edit_distance(x, y) <= max(len(x), len(y))

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. The 0/1 knapsack rolling row must iterate capacity downward. Upward is unbounded knapsack.
2. A rolling array carries an ordering contract that nothing in the code enforces.
3. Label the three neighbours before writing a 2D sequence recurrence.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem